# 📊 Reddit Bronze Layer ETL Pipeline

## Overview
Production-grade incremental ETL pipeline that ingests Reddit submission data from HuggingFace's Arctic dataset into a Unity Catalog bronze table for downstream analytics and processing.

---

## 🎯 Business Objective
Ingest the **top 100 highest-scoring Reddit posts per day** into a bronze layer table, providing raw, minimally-processed data for downstream silver and gold layer transformations.

---

## 📊 Architecture

### Bronze Table: `workspace.Reddit_Recon.posts_bronze`
* **Purpose**: Raw daily Reddit submissions with load metadata
* **Schema**: All Arctic dataset fields + metadata columns
  * `load_year`, `load_month`, `load_day`, `load_date`
* **Update Pattern**: 
  * Append daily (100 posts/day)
  * Truncate on month rollover

### Progress Table: `workspace.Reddit_Recon.load_progress`
* **Purpose**: State management for incremental loading
* **Schema**: `year`, `month`, `day`, `last_updated`
* **Behavior**: Single row always containing next day to load

---

## 🔄 Execution Flow

1. **Setup**: Install dependencies (DuckDB)
2. **Initialize**: Import libraries and configure paths
3. **Schema Creation**: Ensure workspace schema exists
4. **Progress Tracking**: Check current state or initialize from 2024-07-01
5. **Data Load**: Query HuggingFace via DuckDB for target day's top 100 posts
6. **Append**: Write to bronze table with metadata
7. **Advance**: Update progress to next day
8. **Month Rollover**: Clear bronze table when month completes
9. **Summary**: Display bronze table statistics

---

## ⚙️ Production Configuration

### Databricks Workflow Setup
```yaml
Job Name:     Reddit Bronze Daily Ingestion
Schedule:     Daily at 2:00 AM UTC (cron: 0 0 2 * * ?)
Compute:      Serverless (CPU)
Timeout:      30 minutes
Max Retries:  2
Notifications: On failure
Concurrency:  1 (no parallel runs)
```

### Features
* ✅ Self-initializing on first run
* ✅ Idempotent (safe to re-run)
* ✅ Automatic month rollover
* ✅ No manual intervention required
* ✅ Handles missing data gracefully

---

## 📈 Data Selection Criteria

* **Score-based**: Top 100 highest-scoring posts per day
* **Content Filter**: SFW only (`over_18 = false OR NULL`)
* **Time Window**: 24-hour UTC day boundaries
* **Source**: HuggingFace `open-index/arctic` dataset

---

## 🔍 Monitoring

Run the final summary cell to view:
* Daily post counts
* Score ranges (min/max)
* Subreddit diversity
* Load history

In [0]:
try:
    import duckdb
    print("✅ DuckDB already installed")
except ImportError:
    print("📦 Installing DuckDB...")
    %pip install duckdb -q
    dbutils.library.restartPython()
    print("✅ DuckDB installed successfully")

In [0]:
import os
import calendar
import datetime
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import duckdb

print("✅ All libraries imported successfully")

In [0]:
HF_REPO = "open-index/arctic"
BRONZE_TABLE = "workspace.Reddit_Recon.posts_bronze"
PROGRESS_TABLE = "workspace.Reddit_Recon.load_progress"
POSTS_PER_DAY = 100
START_YEAR = "2024"
START_MONTH = "07"
START_DAY = 1

print("="*50)
print("⚙️  ETL PIPELINE CONFIGURATION")
print("="*50)
print(f"Source Repository: {HF_REPO}")
print(f"Target Table:      {BRONZE_TABLE}")
print(f"State Table:       {PROGRESS_TABLE}")
print(f"Daily Volume:      {POSTS_PER_DAY} posts/day")
print(f"Initial Load Date: {START_YEAR}-{START_MONTH}-{START_DAY:02d}")
print("="*50)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.Reddit_Recon
COMMENT 'Bronze layer ingestion for Reddit submissions from HuggingFace Arctic dataset';

In [0]:
try:
    progress_df = spark.table(PROGRESS_TABLE)
    current_state = progress_df.first()
    
    print("\n📊 Current Load State")
    print("─" * 50)
    print(f"  Year:         {current_state.year}")
    print(f"  Month:        {current_state.month}")
    print(f"  Day:          {current_state.day}")
    print(f"  Last Updated: {current_state.last_updated}")
    print("─" * 50)
    
except Exception:
    print("\n🆕 Initializing Progress Tracking...")
    
    schema = StructType([
        StructField("year", StringType(), False),
        StructField("month", StringType(), False),
        StructField("day", IntegerType(), False),
        StructField("last_updated", TimestampType(), False)
    ])
    
    initial_data = [(START_YEAR, START_MONTH, START_DAY, datetime.now())]
    progress_df = spark.createDataFrame(initial_data, schema)
    progress_df.write.mode("overwrite").saveAsTable(PROGRESS_TABLE)
    
    print(f"✅ Progress table initialized")
    print(f"   Starting from: {START_YEAR}-{START_MONTH}-{START_DAY:02d}")
    
    current_state = progress_df.first()

In [0]:
year = current_state.year
month = current_state.month
day = current_state.day

print(f"\n📥 LOADING DATA FOR: {year}-{month}-{day:02d}")
print("="*50)

conn = duckdb.connect()

day_start = int(datetime(int(year), int(month), day).timestamp())
day_end = int(datetime(int(year), int(month), day, 23, 59, 59).timestamp())

print(f"  Querying top {POSTS_PER_DAY} posts...\n")

daily_posts_query = f"""
SELECT *
FROM read_parquet('hf://datasets/{HF_REPO}/data/submissions/{year}/{month}/*.parquet')
WHERE (over_18 = false OR over_18 IS NULL)
  AND created_utc >= {day_start}
  AND created_utc <= {day_end}
ORDER BY score DESC
LIMIT {POSTS_PER_DAY}
"""

try:
    df_daily_posts = conn.execute(daily_posts_query).fetch_df()
    
    row_count = len(df_daily_posts)
    print(f"✅ Successfully fetched {row_count:,} posts")
    
    if row_count > 0:
        print("\n📊 Data Summary")
        print("─" * 40)
        print(f"  Score Range:       {df_daily_posts['score'].min():,} to {df_daily_posts['score'].max():,}")
        print(f"  Unique Subreddits: {df_daily_posts['subreddit'].nunique():,}")
        print(f"  Top 5 Subreddits:  {', '.join(df_daily_posts['subreddit'].value_counts().head(5).index.tolist())}")
        print("─" * 40)
    else:
        print(f"\n No data found for {year}-{month}-{day:02d}")
        
except Exception as e:
    print(f"\n❌ ERROR loading data: {e}")
    raise

print("="*50)

In [0]:
if len(df_daily_posts) > 0:
    print("\n💾 APPENDING TO BRONZE TABLE")
    print("="*50)
    
    df_spark = spark.createDataFrame(df_daily_posts)
    
    df_spark = (df_spark
                .withColumn("load_date", F.current_timestamp()))
    
    try:
        existing = spark.table(BRONZE_TABLE)
        df_spark.write.mode("append").saveAsTable(BRONZE_TABLE)
        
        total_rows = spark.table(BRONZE_TABLE).count()
        print(f"✅ Appended {len(df_daily_posts):,} rows")
        print(f"   Total rows in bronze: {total_rows:,}")
        
    except:
        df_spark.write.mode("overwrite").saveAsTable(BRONZE_TABLE)
        print(f"✅ Created bronze table with {len(df_daily_posts):,} rows")
    
    print("="*50)
    
else:
    print(f"\n  Skipping append - no data for {year}-{month}-{day:02d}")

In [0]:
current_year = int(year)
current_month = int(month)
current_day = day

days_in_month = calendar.monthrange(current_year, current_month)[1]

print("\n🔄 PROGRESS UPDATE")
print("="*50)

if current_day < days_in_month:
    next_year = str(current_year)
    next_month = f"{current_month:02d}"
    next_day = current_day + 1
    
    print(f"  Current:  {year}-{month}-{day:02d}")
    print(f"  Next:     {next_year}-{next_month}-{next_day:02d}")
    
    schema = StructType([
        StructField("year", StringType(), False),
        StructField("month", StringType(), False),
        StructField("day", IntegerType(), False),
        StructField("last_updated", TimestampType(), False)
    ])
    
    new_progress = [(next_year, next_month, next_day, datetime.now())]
    progress_df = spark.createDataFrame(new_progress, schema)
    progress_df.write.mode("overwrite").saveAsTable(PROGRESS_TABLE)
    
    print(f"✅ Progress updated to {next_year}-{next_month}-{next_day:02d}")
    
else:
    print(f"\n🎉 Month {year}-{month} COMPLETE!")
    print("─" * 40)

    if current_month == 12:
        next_year = str(current_year + 1)
        next_month = "01"
    else:
        next_year = str(current_year)
        next_month = f"{current_month + 1:02d}"
    
    next_day = 1
    
    print(f" Clearing bronze table for new month...")
    spark.sql(f"TRUNCATE TABLE {BRONZE_TABLE}")
    print(f"✅ Bronze table cleared")
    
    print(f"\n Starting new month: {next_year}-{next_month}")
    
    schema = StructType([
        StructField("year", StringType(), False),
        StructField("month", StringType(), False),
        StructField("day", IntegerType(), False),
        StructField("last_updated", TimestampType(), False)
    ])
    
    new_progress = [(next_year, next_month, next_day, datetime.now())]
    progress_df = spark.createDataFrame(new_progress, schema)
    progress_df.write.mode("overwrite").saveAsTable(PROGRESS_TABLE)
    
    print(f"✅ Progress reset to {next_year}-{next_month}-01")
    print("─" * 40)

print("\n✨ DAILY LOAD COMPLETE!")
print("="*50)

In [0]:
%sql
SELECT 
  DATE(created_at) as post_date,
  COUNT(*) as post_count,
  MIN(score) as min_score,
  MAX(score) as max_score,
  AVG(score) as avg_score,
  COUNT(DISTINCT subreddit) as unique_subreddits,
  COUNT(DISTINCT author) as unique_authors
FROM workspace.Reddit_Recon.posts_bronze
GROUP BY DATE(created_at)
ORDER BY post_date DESC
LIMIT 31;